<a href="https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — Content freshness correlates with CTR improvement

Label source: CTR change is measured from historical GSC data — an observed outcome, not a constructed proxy. This is a strength.

Does the validation carry the claim? Partially. If freshness and CTR are measured over the same window, the direction of causality is unclear — a page may have been refreshed because it was already recovering, not the other way around. A stronger design would use a held-out future window to confirm CTR movement after refresh. The claim is directional and observed, not causal.

Finding 2 — Position tier predicts CTR more strongly than content age

Label source: CTR is directly observed from GSC impressions and clicks — clean and honest.

Does the validation carry the claim? Yes, more confidently. Position is a stable structural signal and the bucket comparison is straightforward. The risk is Simpson's paradox — if content types are unevenly distributed across position tiers, the aggregate finding may not hold within any single content type. A stratified breakdown would strengthen it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
import os
import numpy as np
import pandas as pd
import getpass
import duckdb
import json
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [10]:
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [11]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')")

In [12]:
REL = "hf://datasets/FlyRank/internship-warehouse"

In [13]:
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [14]:
df = con.execute(f"""
    WITH base AS (
        SELECT
            d.content_hash_id,
            d.client_hash_id,
            d.month,
            d.report_date,
            d.gsc_impressions,
            d.gsc_clicks,
            d.gsc_avg_position,
            MAX(d.report_date) OVER (
                PARTITION BY d.content_hash_id, d.client_hash_id
            ) AS max_date
        FROM {TABLES['fact_daily']} d
        WHERE d.month = '2026-03'
          AND d.gsc_data_available = TRUE
    )
    SELECT
        b.content_hash_id,
        b.client_hash_id,
        b.month,
        SUM(b.gsc_impressions)                                       AS impressions_90d,
        SUM(b.gsc_clicks)                                            AS clicks_90d,
        AVG(b.gsc_avg_position)                                      AS avg_position,
        SUM(b.gsc_clicks) / NULLIF(SUM(b.gsc_impressions), 0)       AS ctr,
        SUM(CASE WHEN b.report_date >= (b.max_date - INTERVAL 30 DAYS)
                 THEN b.gsc_impressions ELSE 0 END)                  AS impressions_last30,
        SUM(CASE WHEN b.report_date < (b.max_date - INTERVAL 30 DAYS)
                 THEN b.gsc_impressions ELSE 0 END)                  AS impressions_prev30,
        c.content_type,
        c.word_count,
        c.last_optimized_date,
        DATEDIFF('day', c.last_optimized_date, MAX(b.report_date))  AS days_since_last_update,
        DATEDIFF('day', c.content_created_date, MAX(b.report_date)) AS content_age_days
    FROM base b
    JOIN {TABLES['dim_content']} c
        ON b.content_hash_id = c.content_hash_id
    GROUP BY
        b.content_hash_id, b.client_hash_id, b.month,
        c.content_type, c.word_count,
        c.last_optimized_date, c.content_created_date
""").df()

df["days_since_last_update"] = df["days_since_last_update"].abs()
df["content_age_days"]       = df["content_age_days"].abs()

print(f"Rows loaded: {len(df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 176,738


In [15]:
# Label
df["ctr_opportunity"] = np.where(
    df["avg_position"] <= 10,
    df["ctr"] < df.groupby(
        pd.cut(df["avg_position"], bins=[0, 3, 5, 10, 20, 100])
    )["ctr"].transform("median"),
    False
)
df["is_declining_label"] = (
    (df["impressions_90d"] >= df["impressions_90d"].median()) &
    (df["ctr_opportunity"] == True)
).astype(int)

# Features
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk  = np.asarray(labels)[order[:k]]
    return topk.mean()

features = ["impressions_90d", "clicks_90d", "avg_position",
            "ctr", "days_since_last_update", "content_age_days", "word_count"]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0).values
y = df["is_declining_label"].values

print(f"Positive labels: {y.sum():,} ({y.mean()*100:.1f}%)")

# Stratified split
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=y
)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(train_idx):,} | Test: {len(test_idx):,}")
print(f"Positive rate — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

Positive labels: 3,416 (1.9%)


/tmp/ipykernel_3429/2076878824.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["ctr"] < df.groupby(


Train: 141,390 | Test: 35,348
Positive rate — train: 0.019 | test: 0.019


In [16]:
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                             max_depth=5, random_state=42)
rf.fit(X_train, y_train)
rf_test = rf.predict_proba(X_test)[:, 1]

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
tree_test = tree.predict_proba(X_test)[:, 1]

print(f"P@50 Random Forest: {precision_at_k(rf_test, y_test, 50):.3f}")
print(f"P@50 Decision Tree: {precision_at_k(tree_test, y_test, 50):.3f}")

P@50 Random Forest: 1.000
P@50 Decision Tree: 0.560


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [17]:
numeric_features = ["impressions_90d", "clicks_90d", "avg_position",
                    "ctr", "days_since_last_update", "content_age_days", "word_count"]

print("=== Feature leakage audit ===")
for col in numeric_features:
    print(f"{col:<30} min={df[col].min():.1f}, max={df[col].max():.1f}, "
          f"nulls={df[col].isnull().sum()} — knowable at decision time")

print(f"\nctr_opportunity       derived from historical CTR vs position median only")

# Confirm label not in feature set
assert "is_declining_label" not in features, "LEAKAGE — label in feature set!"
print("\nNo leakage detected")
print("ctr_opportunity derived from historical CTR vs position median only")
print("is_declining_label not in feature set")
print("No future-window columns used")

=== Feature leakage audit ===
impressions_90d                min=1.0, max=617124.0, nulls=0 — knowable at decision time
clicks_90d                     min=0.0, max=5668.0, nulls=0 — knowable at decision time
avg_position                   min=0.0, max=309.0, nulls=0 — knowable at decision time
ctr                            min=0.0, max=1.0, nulls=0 — knowable at decision time
days_since_last_update         min=24.0, max=125.0, nulls=136974 — knowable at decision time
content_age_days               min=0.0, max=494.0, nulls=0 — knowable at decision time
word_count                     min=0.0, max=29341.0, nulls=55315 — knowable at decision time

ctr_opportunity       derived from historical CTR vs position median only

No leakage detected
ctr_opportunity derived from historical CTR vs position median only
is_declining_label not in feature set
No future-window columns used


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest original sentence:
The Random Forest correctly identifies every page that needs a content refresh in the top 50.

Why it is unsafe:

* Correctly identifies" implies ground truth — my label is a proxy, not a confirmed outcome
* Needs a content refresh" implies causality — I have no evidence refresh fixes the problem
* P@50 = 1.000 on a 1.9% positive rate warrants caution — the model may be exploiting a very clean proxy signal, not learning a generalizable pattern

Rewritten in safe language:
The Random Forest assigned the highest priority scores to pages that match the observed pattern of below-median CTR at their position tier — a directional signal associated with underperformance. These scores are decision-support only: they surface candidates for editorial review. All results are measured from one month of historical data and cannot be used to infer causal impact of content refresh actions or guarantee future ranking improvement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.